# manual-chain-forward-and-back — worked example 1: Manual Forward and Backward Through sqrt → reciprocal

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `manual-chain-forward-and-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

In manual backprop, you write the backward pass as the reverse sequence of per-operation gradient functions. For a 2-step chain `b = sqrt(a)`, `c = 1/b`, the forward goes `a → b → c` and the backward goes `dL/dc → dL/db → dL/da`. Each backward function uses either the cached output or the original input, depending on which is simpler to differentiate.

## Worked solution

**Step 1 — forward pass: a → b → c.**
We compute `b = sqrt(a)` then `c = 1/b`. These are two chained operations. We save `b` and `c` because the backward functions will need them.

**Step 2 — reciprocal_back: compute dL/db.**
The derivative of `1/b` with respect to `b` is `-1/b²`. So `dL/db = dL/dc * (-1/b²)`. We use the cached `b` here, not `c`.

**Step 3 — sqrt_back: compute dL/da.**
The derivative of `sqrt(a)` with respect to `a` is `1 / (2*sqrt(a)) = 1 / (2*b)`. So `dL/da = dL/db * (1 / (2*b))`. Again we use the cached output `b` (= sqrt(a)) rather than recomputing the square root.

**Step 4 — verify against autograd.**
We compute the same chain with `requires_grad=True` and call `.backward()` to get PyTorch's gradient. This must match `dL/da` from our manual computation to within floating-point tolerance, confirming we applied the chain rule correctly.

In [ ]:
import torch as t

t.manual_seed(11)

def sqrt_back(grad_out, out, x):
    """d/dx sqrt(x) = 1 / (2*sqrt(x)) = 1 / (2*out)"""
    return grad_out / (2.0 * out)

def reciprocal_back(grad_out, out, x):
    """d/dx (1/x) = -1/x^2; using input x (not out)"""
    return grad_out * (-1.0 / (x ** 2))

def manual_sqrt_reciprocal_chain(a, dL_dc):
    # Forward: a -> b -> c
    b = t.sqrt(a)       # b = sqrt(a)
    c = 1.0 / b         # c = 1/b
    # Backward (reverse order): dL_dc -> dL_db -> dL_da
    dL_db = reciprocal_back(dL_dc, c, b)   # uses input b
    dL_da = sqrt_back(dL_db, b, a)          # uses output b (= sqrt(a))
    return b, c, dL_db, dL_da

# Run it
t.manual_seed(11)
a_val = t.tensor([4.0, 9.0, 16.0])
dL_dc_val = t.tensor([1.0, 1.0, 1.0])

b, c, dL_db, dL_da = manual_sqrt_reciprocal_chain(a_val, dL_dc_val)
print(f"a={a_val}")
print(f"b=sqrt(a)={b}")
print(f"c=1/b={c}")
print(f"dL/db={dL_db}")
print(f"dL/da (manual)={dL_da}")

# Verify with autograd
a_ag = a_val.clone().requires_grad_(True)
c_ag = 1.0 / t.sqrt(a_ag)
loss = (c_ag * dL_dc_val).sum()
loss.backward()
print(f"dL/da (autograd)={a_ag.grad}")
print(f"Match: {t.allclose(dL_da, a_ag.grad, atol=1e-5)}")